# The Deutsch-Jozsa algorithm

Implement the $U_f$ gate for a general black box function $f: \{0,1\}^n \rightarrow \{0,1\}$. We don't want to simulate the auxiliary qubit explicitly, so the action of the $U_f$ gate is an added phase $(-1)^{f(x)}$ on each basis state x. Let's be super clear on what we mean by this: The computational basis consists of $2^n$ states: $\{|00..0\rangle, |00..01\rangle.., |11..1\rangle  \}$. The bitstring $x$ that enters into the black box function is the binary array of length $n$ that describes the state of each qubit for a given basis state. We can also enumerate  the basis states with the integer numbers $x$ running from $0$ to $2^n-1$. Now the bitstring $x$ is simply the list of binary digits of the integer number $x$. This identification of the "index" of the basis state with the corresponding "state" of the qubits are also be crucial for the quantum Fourier transform.

The function `indToState()` provided below will be very helpful for building the $U_f$ gate. Make sure you understand what it does.

Here are some useful modules you might need to solve this problem. 

In [1]:
# load some useful modules

# standard numerics and linear algebra libraries
import numpy as np  
import numpy.linalg as LA
import scipy.linalg as sciLA

# for making plots
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

# measure runtimes
import time as time 

# sparse matrix functions
import scipy.sparse as sparse

# The black box function. Is it constant or balanced??
from black_box import black_box

%matplotlib inline

In [2]:
# some helper functions
def indToState(n, k):
    num = bin(k)[2:].zfill(n)
    return np.array([int(x) for x in str(num)])

# not necessarily needed:
def stateToInd(state):
    return int("".join(str(x) for x in state),2)

Now implement the Deutsch-Jozsa algorithm and use it to detemine for which $n$ the black box function provided along with this problem set is constant and for which it is balanced!

Check the result "classically". 

Don't look at the source code of the black box function! This will make the wave function of the universe collapse ;-)

### Solution

In [7]:
import qutip as qt
from circuit_engine import apply_instruction, measure_qubit
from deutsch_jozsa import create_dynamic_oracle

In [ ]:
def run_quantum_blackbox_test(max_n: int):
    
    for n in range(1, max_n + 1):
        # 1. Initialize |00...0>
        state = qt.tensor([qt.basis(2, 0)] * n)
        
        # 2. Superposition
        for q in range(n):
            state = apply_instruction(state, ['H', [q]])
            
        # 3. Apply Dynamic Oracle
        U_f = create_dynamic_oracle(n, black_box)
        state = U_f * state
        
        # 4. Interference
        for q in range(n):
            state = apply_instruction(state, ['H', [q]])
            
        # 5. Measure
        measurements = []
        for q in range(n):
            result, state = measure_qubit(state, target_qubit=q)
            measurements.append(result)
            
        # 6. Evaluate
        if sum(measurements) == 0:
            print(f"For n={n} qubits: CONSTANT (Measured all zeros)")
        else:
            print(f"For n={n} qubits: BALANCED (Measured {measurements})")


run_quantum_blackbox_test(max_n=6)

=== Quantum Blackbox Discovery ===
For n=1 qubits: BALANCED (Measured [1])
For n=2 qubits: BALANCED (Measured [0, 1])
For n=3 qubits: CONSTANT (Measured all zeros)
For n=4 qubits: BALANCED (Measured [0, 0, 1, 0])
For n=5 qubits: CONSTANT (Measured all zeros)
For n=6 qubits: CONSTANT (Measured all zeros)


In [10]:
def classical_check(n: int):
    dim = 2 ** n
    outputs = []
    
    print(f"\nClassical Check for n={n}:")
    for k in range(dim):
        bit_string = indToState(n, k)
        result = black_box(bit_string)
        outputs.append(result)
        
    if all(x == 0 for x in outputs) or all(x == 1 for x in outputs):
        print(f"Classical Conclusion: CONSTANT. (Looked at all {dim} outputs: {outputs})")
    else:
        print(f"Classical Conclusion: BALANCED. (Looked at all {dim} outputs: {outputs})")

# Run this to check your quantum answers!
for n in range(1, 7):
    classical_check(n)


Classical Check for n=1:
Classical Conclusion: BALANCED. (Looked at all 2 outputs: [0, 1])

Classical Check for n=2:
Classical Conclusion: BALANCED. (Looked at all 4 outputs: [0, 1, 0, 1])

Classical Check for n=3:
Classical Conclusion: CONSTANT. (Looked at all 8 outputs: [0, 0, 0, 0, 0, 0, 0, 0])

Classical Check for n=4:
Classical Conclusion: BALANCED. (Looked at all 16 outputs: [0, 0, 1, 1, 0, 0, 1, 1, 0, 0, 1, 1, 0, 0, 1, 1])

Classical Check for n=5:
Classical Conclusion: CONSTANT. (Looked at all 32 outputs: [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1])

Classical Check for n=6:
Classical Conclusion: CONSTANT. (Looked at all 64 outputs: [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0])
